# 3.4 Data Integration

> **Project:** Titanic Survival Prediction
> **Date:** 2026-03-31
> **CRISP-DM Phase:** 3. Data Preparation — Task 3.4

## Purpose

This notebook documents the data integration assessment for the Titanic project. Since the project uses a **single data source** (Kaggle Titanic dataset), no multi-source integration is required. This notebook validates that the existing pipeline outputs are complete and ready for modeling.

In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(__file__).resolve().parent.parent if "__file__" in dir() else Path.cwd()
if (PROJECT_ROOT / "notebooks").is_dir():
    pass  # cwd is project root
elif (PROJECT_ROOT.parent / "notebooks").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent  # cwd is a subdirectory

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

## Integration Assessment

**Conclusion: No multi-source integration required.**

The Titanic project has a single data source (Kaggle) split into train/test sets with identical schemas (minus the `Survived` target). The pipeline processes them in parallel through the same stages:

1. **3.1 Select Data** — selected `train.csv` (891 rows) and `test.csv` (418 rows)
2. **3.2 Clean Data** — imputed missing values → `train_clean.csv`, `test_clean.csv`
3. **3.3 Construct Data** — engineered features → `train_features.csv`, `test_features.csv`

The only cross-dataset operation (TicketGroupSize computed on combined train+test) was already handled in task 3.3.

## Validation: Pipeline Output Completeness

Load the final feature-engineered datasets and verify they are complete, schema-aligned, and ready for modeling.

In [2]:
train = pd.read_csv(PROCESSED_DIR / "train_features.csv")
test = pd.read_csv(PROCESSED_DIR / "test_features.csv")

print(f"Train: {train.shape[0]} rows x {train.shape[1]} cols")
print(f"Test:  {test.shape[0]} rows x {test.shape[1]} cols")
print(f"\nTrain columns: {sorted(train.columns.tolist())}")
print(f"Test columns:  {sorted(test.columns.tolist())}")

# Identify schema differences (expected: only Survived)
train_only = set(train.columns) - set(test.columns)
test_only = set(test.columns) - set(train.columns)
print(f"\nColumns in train only: {train_only}")
print(f"Columns in test only:  {test_only}")

Train: 891 rows x 22 cols
Test:  418 rows x 21 cols

Train columns: ['Age', 'AgeMissing', 'Cabin', 'Deck', 'Embarked_Q', 'Embarked_S', 'FamilySize', 'FamilySizeBin', 'Fare', 'FareLog', 'HasCabin', 'IsAlone', 'Name', 'Parch', 'PassengerId', 'Pclass', 'Sex', 'SibSp', 'Survived', 'Ticket', 'TicketGroupSize', 'Title']
Test columns:  ['Age', 'AgeMissing', 'Cabin', 'Deck', 'Embarked_Q', 'Embarked_S', 'FamilySize', 'FamilySizeBin', 'Fare', 'FareLog', 'HasCabin', 'IsAlone', 'Name', 'Parch', 'PassengerId', 'Pclass', 'Sex', 'SibSp', 'Ticket', 'TicketGroupSize', 'Title']

Columns in train only: {'Survived'}
Columns in test only:  set()


In [3]:
# Check for missing values in the feature columns (excluding target)
feature_cols = sorted(set(train.columns) & set(test.columns))

print("Missing values in feature columns:")
print(f"\n{'Column':<20} {'Train':>8} {'Test':>8}")
print("-" * 38)
for col in feature_cols:
    t_miss = train[col].isna().sum()
    te_miss = test[col].isna().sum()
    if t_miss > 0 or te_miss > 0:
        print(f"{col:<20} {t_miss:>8} {te_miss:>8}")

train_missing = train[feature_cols].isna().sum().sum()
test_missing = test[feature_cols].isna().sum().sum()
print(f"\nTotal missing in shared features: train={train_missing}, test={test_missing}")

Missing values in feature columns:

Column                  Train     Test
--------------------------------------
Cabin                     687      327

Total missing in shared features: train=687, test=327


In [4]:
# Verify row counts match expectations
assert train.shape[0] == 891, f"Expected 891 train rows, got {train.shape[0]}"
assert test.shape[0] == 418, f"Expected 418 test rows, got {test.shape[0]}"

# Verify PassengerId ranges are disjoint
assert set(train["PassengerId"]).isdisjoint(set(test["PassengerId"])), "PassengerId overlap detected!"

# Verify target exists only in train
assert "Survived" in train.columns, "Survived missing from train"
assert "Survived" not in test.columns, "Survived should not be in test"

print("All validation checks passed:")
print("  - Train has 891 rows, test has 418 rows")
print("  - PassengerId ranges are disjoint")
print("  - Survived present in train only")
print("\nDatasets are ready for modeling (Phase 4).")

All validation checks passed:
  - Train has 891 rows, test has 418 rows
  - PassengerId ranges are disjoint
  - Survived present in train only

Datasets are ready for modeling (Phase 4).
